# Lesson 2 · NumPy Superpowers: Vectorization, Broadcasting, Matrices & Randomness
### Stop writing loops. Start thinking in whole arrays.

| | |
|---|---|
| **Type** | LESSON 02 |
| **Time** | about 120 minutes |
| **Level** | Beginner → Intermediate |
| **Prerequisites** | Lesson 1 (arrays, indexing, slicing, masks) |

**By the end of this notebook you will be able to:**

- Replace Python loops with vectorised operations and *measure* the gain
- Summarise data with aggregations along any axis
- Predict the result (or the error!) of any broadcasting operation using the two shape rules
- Use matrix multiplication, inverses and linear solvers on real questions
- Handle floating-point quirks, NaNs, differences and percentage changes
- Generate reproducible random numbers, run a Monte-Carlo simulation and bootstrap an average

## 🛠️ Setup
Run this cell first. It imports the tools and loads the data every problem below relies on.

In [1]:

import sys; sys.path.insert(0, "..")
import time
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check

rng = np.random.default_rng(7)
# 8 weeks x 3 crops of weekly trading volume (tonnes): maize, beans, tomatoes
weekly_volume = np.round(rng.normal([50, 120, 30], [5, 15, 4], size=(8, 3)), 1)
# Maize price (KES/kg): 3 markets x 4 weeks, and the KES-per-USD exchange rate in each of those weeks
price_kes = np.array([[48.0, 49.5, 51.0, 50.2],
                      [52.0, 53.5, 55.0, 54.1],
                      [45.5, 46.0, 47.2, 46.8]])
usd_rate = np.array([128.5, 129.0, 130.2, 131.0])
# A short price series
series = np.array([50, 52, 51, 55, 54, 60, 57, 63, 61, 66])
# Real weekly maize price & volume at Wakulima Market
df = pd.read_csv(DATA_DIR / "market_prices_clean.csv")
_m = (df["crop"] == "Maize") & (df["market"] == "Wakulima Market")
mz_price = df.loc[_m, "price_kes_per_kg"].to_numpy()
mz_vol = df.loc[_m, "volume_kg"].to_numpy()
print("Setup complete ✔")


Setup complete ✔


## 👋 Where we left off

In Lesson 1 you learned to *point at* data. Today you learn to *compute on* data, and I want to plant one big idea in your head:

> ## **Think in arrays, not in elements.**
> A beginner asks, *"For each price, do this…"* and reaches for a `for` loop.
> A data scientist asks, *"What do I want to do to the whole column?"* and writes one line.

Our running story: your market-information service now wants **a price index**, **a USD conversion**, **a weekly volatility report**
and a **forecast simulation**. Let's build them all without a single loop.

## 1 · Vectorisation: operations on whole arrays

An operation is **vectorised** when it is applied to every element at once. Arithmetic operators already work that way:

In [2]:
p = np.array([50.0, 52.5, 55.0, 53.0])
print("p + 10      =", p + 10)             # a scalar is applied to every element
print("p * 1.16    =", p * 1.16)           # e.g. add 16% VAT
print("p / 130     =", (p / 130).round(3)) # convert KES to USD
print("p ** 2      =", p ** 2)
q = np.array([2, 4, 6, 8])
print("p * q       =", p * q)              # two same-shape arrays -> element by element

p + 10      = [60.  62.5 65.  63. ]
p * 1.16    = [58.   60.9  63.8  61.48]
p / 130     = [0.385 0.404 0.423 0.408]
p ** 2      = [2500.   2756.25 3025.   2809.  ]
p * q       = [100. 210. 330. 424.]


**Universal functions ("ufuncs")** apply a mathematical function element-wise: `np.sqrt`, `np.exp`, `np.log`, `np.abs`, `np.sin`,
`np.maximum`, `np.clip`, …

In [3]:
print("log     :", np.log(p).round(3))
print("sqrt    :", np.sqrt(p).round(3))
print("clip    :", np.clip(p, 51, 54))                # force values into [51, 54]
print("maximum :", np.maximum(p, 53))                 # element-wise "larger of the two"

log     : [3.912 3.961 4.007 3.97 ]
sqrt    : [7.071 7.246 7.416 7.28 ]
clip    : [51.  52.5 54.  53. ]
maximum : [53. 53. 55. 53.]


### Loop vs vectorised: a fair fight

Task: convert **two million** KES prices to USD, rounding to cents.

In [4]:
big = np.random.default_rng(0).uniform(30, 150, 2_000_000)

t0 = time.perf_counter()
usd_loop = []
for x in big:                                   # the "beginner" way
    usd_loop.append(round(x / 129.0, 2))
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
usd_vec = np.round(big / 129.0, 2)              # the "data scientist" way
t_vec = time.perf_counter() - t0

print(f"loop      : {t_loop:.3f} s")
print(f"vectorised: {t_vec:.3f} s   ->  about {t_loop/t_vec:.0f}x faster")
print("same answer?", np.allclose(usd_loop, usd_vec))

loop      : 4.737 s
vectorised: 0.015 s   ->  about 310x faster
same answer? True


> 💡 **Intuition.** A loop is a cashier scanning items one by one. Vectorisation is a conveyor belt feeding an automatic scanner. Same shop, same groceries, wildly different queue.

> ⚠️ **Common mistake.** `if` statements don't vectorise: `if p > 50:` fails on an array. Use boolean masks or `np.where(p > 50, a, b)` (Lesson 1).

## 2 · Aggregations along an axis

Reductions collapse an array to fewer numbers. Memorise this family:

| Function | Meaning | Function | Meaning |
|---|---|---|---|
| `sum` | total | `min`, `max` | extremes |
| `mean` | average | `argmin`, `argmax` | position of extremes |
| `median` | middle value | `cumsum`, `cumprod` | running total / product |
| `std`, `var` | spread | `percentile`, `quantile` | cut-points |

In [5]:
print("weekly_volume shape:", weekly_volume.shape, "(8 weeks x 3 crops)")
print("Total per crop  (axis=0):", weekly_volume.sum(axis=0))
print("Mean per crop   (axis=0):", weekly_volume.mean(axis=0).round(1))
print("Std per crop    (axis=0):", weekly_volume.std(axis=0).round(1))
print("Total per week  (axis=1):", weekly_volume.sum(axis=1).round(1))
print("Running total of maize  :", weekly_volume[:, 0].cumsum().round(1))
print("25th, 50th, 75th percentile of maize:", np.percentile(weekly_volume[:, 0], [25, 50, 75]).round(1))

weekly_volume shape: (8, 3) (8 weeks x 3 crops)
Total per crop  (axis=0): [386.  912.6 226.1]
Mean per crop   (axis=0): [ 48.3 114.1  28.3]
Std per crop    (axis=0): [ 3.7 14.1  2.7]
Total per week  (axis=1): [203.4 184.7 218.4 205.6 186.4 181.5 163.8 180.9]
Running total of maize  : [ 50.   95.5 145.8 192.7 243.2 296.7 337.2 386. ]
25th, 50th, 75th percentile of maize: [46.6 49.4 50.3]


> 📝 **Note.** `std` here is the **population** standard deviation (`ddof=0`). Pandas' `.std()` defaults to the **sample** version (`ddof=1`). This tiny difference trips people up when comparing NumPy and pandas results: pass `ddof=1` to NumPy to match pandas.

`keepdims=True` keeps the collapsed axis as size 1; extremely handy for the next section:

In [6]:
m = weekly_volume.mean(axis=0, keepdims=True)
print("shape without keepdims:", weekly_volume.mean(axis=0).shape, "| with keepdims:", m.shape)

shape without keepdims: (3,) | with keepdims: (1, 3)


## 3 · Broadcasting: NumPy's cleverest trick

What happens when you combine arrays of **different shapes**? NumPy *stretches* the smaller one so they fit, without copying data.

### The two rules

Line up the shapes **from the right**. For each pair of dimensions:

1. If they are **equal**, fine.
2. If one of them is **1**, it is stretched to match the other.

Otherwise: `ValueError: operands could not be broadcast together`.

```
price_kes   (3, 4)         price_kes   (3, 4)         price_kes (3, 4)
usd_rate       (4,)   ✔   adj_per_market (3,)  ✘     adj (3, 1)    ✔
-> result   (3, 4)         3 vs 4 → mismatch!         -> result (3, 4)
```

In [7]:
# Convert every market's price to USD, using that week's exchange rate (one rate per COLUMN)
price_usd = price_kes / usd_rate
print(price_kes.shape, "/", usd_rate.shape, "->", price_usd.shape)
print(price_usd.round(3))

(3, 4) / (4,) -> (3, 4)
[[0.374 0.384 0.392 0.383]
 [0.405 0.415 0.422 0.413]
 [0.354 0.357 0.363 0.357]]


> 🧠 **Pause and predict.** Suppose each *market* (row) charges a different transport markup: `[1.0, 2.5, 0.5]`. Will `price_kes + markup` work? Predict first!

In [8]:
markup = np.array([1.0, 2.5, 0.5])
try:
    price_kes + markup
except ValueError as err:
    print("ValueError:", err)          # shapes (3,4) and (3,) -> compare from the right: 4 vs 3 ✘

print("\nFix: give markup a column shape (3, 1):")
markup_col = markup[:, np.newaxis]     # same as markup.reshape(3, 1)
print((price_kes + markup_col).round(1))

ValueError: operands could not be broadcast together with shapes (3,4) (3,) 

Fix: give markup a column shape (3, 1):
[[49.  50.5 52.  51.2]
 [54.5 56.  57.5 56.6]
 [46.  46.5 47.7 47.3]]


> 💡 **Intuition.** When broadcasting goes wrong, **print the shapes** and line them up from the right. A 1-D array of length *n* behaves like a **row** `(1, n)`; to make it a **column** you must add an axis: `(n, 1)`.

### Broadcasting in action: normalising data

A very common ML preparation step is **standardising** each column: subtract its mean and divide by its standard deviation
(the *z-score*). Broadcasting makes it a one-liner because `mean(axis=0)` has shape `(3,)` and the data has shape `(8, 3)`.

In [9]:
mean = weekly_volume.mean(axis=0)            # shape (3,)
std = weekly_volume.std(axis=0)              # shape (3,)
z = (weekly_volume - mean) / std             # (8,3) - (3,) -> (8,3)
print("column means after:", z.mean(axis=0).round(6))
print("column stds  after:", z.std(axis=0).round(6))

column means after: [-0.  0.  0.]
column stds  after: [1. 1. 1.]


> 🤖 **Why machine learning cares.** Features on different scales (volume in hundreds, price in tens) can dominate distance-based models such as k-NN and k-means. Standardising is a routine step. You just learned to do it by hand.

**Outer operations with `newaxis`.** Want every combination of 3 markups × 4 base prices? Make one a column and one a row:

In [10]:
base = np.array([40, 50, 60, 70])
mk = np.array([1, 2, 3])
table = base[np.newaxis, :] + mk[:, np.newaxis]     # (1,4) + (3,1) -> (3,4)
print(table)

[[41 51 61 71]
 [42 52 62 72]
 [43 53 63 73]]


## 4 · Matrix operations

A 2-D array can be treated as a mathematical **matrix**. Two very different "multiplications" exist:

* `A * B`: **element-wise** (same positions multiplied). Needs broadcast-compatible shapes.
* `A @ B`: **matrix product** (rows × columns, summed). Needs inner dimensions to match: `(m, n) @ (n, p) → (m, p)`.

**Example.** Three markets quoted prices for three crops. Our team sold a known quantity of each crop. What's the revenue at each market's prices?

In [11]:
prices_tbl = np.array([[60, 150, 90],       # market A:  maize, beans, tomatoes (KES/kg)
                       [65, 160, 95],       # market B
                       [58, 145, 85]])      # market C
kg_sold = np.array([1000, 400, 700])        # kg of each crop

revenue = prices_tbl @ kg_sold               # (3,3) @ (3,) -> (3,)
print("Revenue if sold at A, B, C prices:", revenue)

# What the @ did for market A, written out:
print("Check market A by hand:", 60*1000 + 150*400 + 90*700)

Revenue if sold at A, B, C prices: [183000 195500 175500]
Check market A by hand: 183000


### The matrix toolbox

| Task | Code |
|---|---|
| Transpose | `A.T` |
| Matrix product | `A @ B` (or `np.dot`) |
| Identity | `np.eye(n)` |
| Determinant | `np.linalg.det(A)` |
| Inverse | `np.linalg.inv(A)` |
| Solve `A x = b` | `np.linalg.solve(A, b)` |
| Least squares | `np.linalg.lstsq(X, y, rcond=None)` |

**Solving equations.** Two traders each bought maize (m) and beans (b) by the kilogram:

* Trader 1: 2 kg maize + 1 kg beans = 245 KES
* Trader 2: 1 kg maize + 3 kg beans = 485 KES

What are the per-kg prices? Write as `A @ x = b` and let NumPy solve it. (Prefer `solve` over `inv(A) @ b`: it's faster and more accurate.)

In [12]:
A = np.array([[2, 1],
              [1, 3]])
b = np.array([245, 485])
x = np.linalg.solve(A, b)
print("maize =", x[0], "KES/kg   beans =", x[1], "KES/kg")
print("verify A @ x == b ?", np.allclose(A @ x, b))

maize = 50.0 KES/kg   beans = 145.0 KES/kg
verify A @ x == b ? True


### 🔭 A glimpse of machine learning: linear regression in four lines

*"How much does rainfall lower tomato prices?"* The model is **price ≈ β₀ + β₁ · rainfall**. Put a column of ones next to the rainfall
values (that's what lets the line have an intercept) and let least-squares find the best β:

In [13]:
r = np.random.default_rng(1)
rain = r.uniform(20, 300, 60)                                   # mm of rain in a month
price_t = 90 - 0.08 * rain + r.normal(0, 3, 60)                 # "truth": each extra mm lowers price by 0.08

X = np.column_stack([np.ones_like(rain), rain])                  # design matrix [1, rain]
beta, *_ = np.linalg.lstsq(X, price_t, rcond=None)
print(f"Estimated intercept = {beta[0]:.2f}  (truth 90)")
print(f"Estimated slope     = {beta[1]:.3f} (truth -0.08)")
print("Prediction for 150 mm of rain:", round(float(beta[0] + beta[1] * 150), 1), "KES/kg")

Estimated intercept = 89.23  (truth 90)
Estimated slope     = -0.077 (truth -0.08)
Prediction for 150 mm of rain: 77.7 KES/kg


> 🤖 **Why machine learning cares.** Training a linear model *is* solving a matrix problem. In Phase 2 you will use scikit-learn, but now you know what's under the hood.

## 5 · Numerical operations you'll use every day

### 5.1 Floating-point honesty
Computers store decimals in binary, so some numbers are *slightly* off. Never compare floats with `==`:

In [14]:
print(0.1 + 0.2 == 0.3, "  <- surprise!")
print(np.isclose(0.1 + 0.2, 0.3), " <- use isclose / allclose")
print(np.round(2.675, 2), np.floor(2.9), np.ceil(2.1), np.trunc(-2.9))

False   <- surprise!
True  <- use isclose / allclose
2.68 2.0 3.0 -2.0


### 5.2 Missing values (`NaN`) are contagious

In [15]:
with_gap = np.array([50.0, 52.0, np.nan, 55.0])
print("sum    :", with_gap.sum(), " <- one NaN poisons the answer")
print("nansum :", np.nansum(with_gap), "| nanmean:", np.nanmean(with_gap).round(2))
print("which are NaN?", np.isnan(with_gap), "| how many?", np.isnan(with_gap).sum())
print("NaN == NaN ?", np.nan == np.nan, " <- always False; use np.isnan")

sum    : nan  <- one NaN poisons the answer
nansum : 157.0 | nanmean: 52.33
which are NaN? [False False  True False] | how many? 1
NaN == NaN ? False  <- always False; use np.isnan


### 5.3 Changes over time: `np.diff`, percentage change, cumulative sums

In [16]:
s = series.astype(float)
change = np.diff(s)                       # s[1]-s[0], s[2]-s[1], ...  (one element shorter!)
pct = change / s[:-1] * 100               # percentage change vs the previous period
print("series          :", s)
print("weekly change   :", change)
print("weekly % change :", pct.round(1))
print("largest weekly jump: week", pct.argmax() + 1, "->", pct.max().round(1), "%")

series          : [50. 52. 51. 55. 54. 60. 57. 63. 61. 66.]
weekly change   : [ 2. -1.  4. -1.  6. -3.  6. -2.  5.]
weekly % change : [ 4.  -1.9  7.8 -1.8 11.1 -5.  10.5 -3.2  8.2]
largest weekly jump: week 5 -> 11.1 %


> ⚠️ **Common mistake.** `np.diff` returns **n−1** values. Line it up carefully with your dates: change number *i* belongs to week *i+1*. Off-by-one slips here silently ruin forecasts.

### 5.4 Spotting outliers with percentiles (the IQR rule)

The **interquartile range** `IQR = Q3 − Q1` measures the spread of the middle 50%. A common rule flags anything beyond
`Q1 − 1.5·IQR` or `Q3 + 1.5·IQR`. (It's the same rule behind box-plot whiskers, which we draw in Lesson 6.)

In [17]:
data = np.array([52, 53, 51, 55, 54, 56, 53, 52, 540, 55, 54, 5.3])   # two data-entry slips: 540 and 5.3
q1, q3 = np.percentile(data, [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}  ->  acceptable range [{low:.1f}, {high:.1f}]")
print("Outliers:", data[(data < low) | (data > high)])

Q1=52.0, Q3=55.0, IQR=3.0  ->  acceptable range [47.5, 59.5]
Outliers: [540.    5.3]


### 5.5 Counting into bins and correlation

In [18]:
counts, edges = np.histogram(mz_price, bins=5)
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"{lo:5.1f} – {hi:5.1f} KES : {'█' * (c // 4)} {c}")

print("\nCorrelation between maize price and volume (Wakulima):", np.corrcoef(mz_price, mz_vol)[0, 1].round(3))

 44.8 –  52.0 KES : █████████ 39
 52.0 –  59.3 KES : ████████████ 51
 59.3 –  66.5 KES : ██████████ 41
 66.5 –  73.8 KES : ████████████ 51
 73.8 –  81.0 KES : ██████ 27

Correlation between maize price and volume (Wakulima): -0.406


> 🧠 **Pause and predict.** The correlation is negative. Why might a *higher price* go with *lower volume traded*? Think supply and demand. We'll return to this in the EDA lesson.

## 6 · Random numbers

Randomness powers simulation, sampling, train/test splits and neural-network initialisation. The modern NumPy way: create a
**generator** once, then ask it for numbers.

```python
rng = np.random.default_rng(seed=123)     # the seed makes results reproducible
```

In [19]:
rng_demo = np.random.default_rng(123)
print("uniform [0,1)   :", rng_demo.random(3).round(3))
print("integers 1..6   :", rng_demo.integers(1, 7, size=5))              # a dice; the upper bound is EXCLUDED
print("normal(100,15)  :", rng_demo.normal(100, 15, size=3).round(1))
print("choice w/o repl :", rng_demo.choice(["Maize", "Beans", "Kale", "Onions"], size=2, replace=False))
print("permutation     :", rng_demo.permutation(5))

same_again = np.random.default_rng(123)
print("\nSame seed, same numbers:", same_again.random(3).round(3))

uniform [0,1)   : [0.682 0.054 0.22 ]
integers 1..6   : [2 2 3 2 3]
normal(100,15)  : [ 90.5 108.1  95.3]
choice w/o repl : ['Kale' 'Onions']
permutation     : [3 2 0 1 4]

Same seed, same numbers: [0.682 0.054 0.22 ]


> 💡 **Intuition.** The seed is the starting position in a very long, shuffled-looking list. Same seed → same list → **reproducible science**. Always seed anything you want a colleague (or future you) to be able to re-run.

> 📝 **Note.** You may see old tutorials using `np.random.seed(0)` and `np.random.rand()`. That legacy global API still works, but `default_rng` is the recommended modern approach.

### Which distribution? A tiny guide

| Situation | Distribution | Call |
|---|---|---|
| Any value in a range equally likely | uniform | `rng.uniform(a, b, n)` |
| Bell-shaped measurement noise | normal | `rng.normal(mean, sd, n)` |
| Prices & incomes (positive, right-skewed) | log-normal | `rng.lognormal(mu, sigma, n)` |
| Counts of rare events (breakdowns per week) | Poisson | `rng.poisson(lam, n)` |
| Yes/No with a probability | binomial | `rng.binomial(1, p, n)` |

---
# 🧑‍🏫 Worked examples

## Worked example 1: Build a price index (broadcasting)

> **Question.** Using `price_kes` (3 markets × 4 weeks), compute each market's **price index** where week 0 = 100.
> Then report which market's price rose the most by week 3.

**Thinking steps**
1. Base for each market = its week-0 price = `price_kes[:, 0]`, shape `(3,)`.
2. We must divide **each row** by its own base. Rows need a `(3, 1)` column, so add an axis (or slice `[:, 0:1]`).
3. Multiply by 100. The index at week 3 tells the story.

In [20]:
base = price_kes[:, 0:1]                  # shape (3, 1): keeps the column shape
index = price_kes / base * 100            # (3,4) / (3,1) -> (3,4)
print(index.round(1))
growth = index[:, -1] - 100
print("Growth by week 3 (%):", growth.round(1), "-> biggest riser is market", growth.argmax())

[[100.  103.1 106.2 104.6]
 [100.  102.9 105.8 104. ]
 [100.  101.1 103.7 102.9]]
Growth by week 3 (%): [4.6 4.  2.9] -> biggest riser is market 0


## Worked example 2: Monte-Carlo: "What's the chance price tops 60?"

> **Question.** Maize sells at 50 KES/kg today. Each week the price moves by a random amount ~ Normal(mean 0, sd 1.5).
> What is the probability the price **exceeds 60 after 12 weeks**?

**Thinking steps**
1. One simulated future = 12 random steps summed onto 50.
2. Simulate **20,000 futures at once** with a `(20000, 12)` array, sum along `axis=1`.
3. The proportion of futures above 60 ≈ the probability (mean of a Boolean mask!).

Theory check: the sum of 12 steps has sd = 1.5·√12 ≈ 5.2, so 60 is ≈ 1.9 sd above 50 → probability ≈ 3%.

In [21]:
sim = np.random.default_rng(2024)
steps = sim.normal(0, 1.5, size=(20_000, 12))
final = 50 + steps.sum(axis=1)
print("Mean final price :", final.mean().round(2))
print("Std of final     :", final.std().round(2), "(theory 5.20)")
print("P(price > 60)    :", (final > 60).mean().round(4))
print("90% of futures lie between", np.percentile(final, [5, 95]).round(1))

Mean final price : 50.0
Std of final     : 5.24 (theory 5.20)
P(price > 60)    : 0.0296
90% of futures lie between [41.5 58.7]


## Worked example 3: Bootstrap: how sure are we about an average?

> **Question.** Using the real Wakulima maize prices (`mz_price`), give a 95% interval for the **mean price**.

**Idea.** Resample the data *with replacement* many times, compute the mean each time, and look at the spread of those means.

*(Honest caveat: weekly prices are correlated with each other over time, which this simple bootstrap ignores, so the true uncertainty is wider than shown. Knowing a method's assumptions is half of using it well.)*

In [22]:
boot = np.random.default_rng(5)
means = np.array([boot.choice(mz_price, size=mz_price.size, replace=True).mean() for _ in range(2000)])
lo, hi = np.percentile(means, [2.5, 97.5])
print(f"Sample mean = {mz_price.mean():.2f} KES/kg   95% bootstrap interval: [{lo:.2f}, {hi:.2f}]")

Sample mean = 61.89 KES/kg   95% bootstrap interval: [60.67, 63.20]


---
# 🏋️ Problems

All problems start from the **Setup** cell variables. Auto-checks accept small rounding differences, but follow the stated rounding.

### 🧩 Problem 2.1: Standardise the volumes

Standardise each **column** of `weekly_volume` to a z-score: `(x − column mean) / column standard deviation` (use NumPy's default `std`). Return the **first row** of the standardised matrix, rounded to 2 decimals.

*Expected type:* an array of 3 numbers.

<details>
<summary>💡 Hint 1</summary>

You need statistics *per column*: which axis do you aggregate over to get one value per column?

</details>

<details>
<summary>💡 Hint 2</summary>

A `(8, 3)` array minus a `(3,)` array broadcasts perfectly: no loop required.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`z = (weekly_volume - weekly_volume.mean(axis=0)) / weekly_volume.std(axis=0)`, then `np.round(z[0], 2)`.

</details>

*Auto-check id:* `02-p1`  ·  *Set the variable `answer`, then run the cell.*

In [23]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('02-p1', answer)

⏳ 02-p1: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 2.2: Convert to dollars, week by week

`price_kes` has shape `(3, 4)` and `usd_rate` has shape `(4,)` (KES per USD in each week). Return the USD price matrix (KES price ÷ rate), rounded to **3 decimals**.

*Expected type:* a `(3, 4)` array.

<details>
<summary>💡 Hint 1</summary>

Check the shapes from the right: `(3, 4)` and `(4,)`. Do they broadcast?

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

If they do, the operation is a single division. Finish with `np.round(..., 3)`.

</details>

*Auto-check id:* `02-p2`  ·  *Set the variable `answer`, then run the cell.*

In [24]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('02-p2', answer)

⏳ 02-p2: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 2.3: The biggest weekly jump

For the array `series`, compute the **percentage change** from each week to the next and return the **largest** one, rounded to 2 decimals.

*Expected type:* float (a percentage such as `7.84`).

<details>
<summary>💡 Hint 1</summary>

`np.diff(series)` gives the absolute changes (one element shorter).

</details>

<details>
<summary>💡 Hint 2</summary>

Divide by the *previous* values: `series[:-1]`. Multiply by 100.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`np.round((np.diff(series) / series[:-1] * 100).max(), 2)`

</details>

*Auto-check id:* `02-p3`  ·  *Set the variable `answer`, then run the cell.*

In [25]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('02-p3', answer)

⏳ 02-p3: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 2.4: Which prices?

A trader pays **5 kg of maize + 2 kg of beans = 600 KES**, and another pays **3 kg of maize + 4 kg of beans = 780 KES**. Use a NumPy linear solver to find the per-kg prices.

*Expected type:* an array `[maize_price, beans_price]`.

<details>
<summary>💡 Hint 1</summary>

Write the left-hand coefficients as a 2×2 matrix `A` and the totals as a vector `b`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`np.linalg.solve(A, b)` returns `x` such that `A @ x = b`.

</details>

*Auto-check id:* `02-p4`  ·  *Set the variable `answer`, then run the cell.*

In [26]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('02-p4', answer)

⏳ 02-p4: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 2.5: Tight-market weeks (no loops!)

`mz_price` and `mz_vol` hold weekly maize price and volume at Wakulima. A week is **tight** if its price is **above the median price** *and* its volume is **below the median volume**. How many tight weeks are there?

*Expected type:* an integer.

<details>
<summary>💡 Hint 1</summary>

Two conditions → two Boolean masks combined with `&` (parentheses!).

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`np.median(...)` gives each benchmark. Finish by summing the combined mask.

</details>

*Auto-check id:* `02-p5`  ·  *Set the variable `answer`, then run the cell.*

In [27]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('02-p5', answer)

⏳ 02-p5: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 2.6: Will the price spike? (Monte-Carlo)

A price starts at 50 and each of the next 12 weeks adds a random step drawn from Normal(mean 0, sd 2). Using `rng = np.random.default_rng(7)` and **exactly** `steps = rng.normal(0, 2, size=(10000, 12))`, simulate 10,000 futures. Return the **proportion** whose final price (50 + sum of steps) is **above 55**.

*Expected type:* float between 0 and 1.

<details>
<summary>💡 Hint 1</summary>

Start with `rng = np.random.default_rng(7)` then call `rng.normal(0, 2, size=(10000, 12))`: in that order, so your random numbers match ours.

</details>

<details>
<summary>💡 Hint 2</summary>

Sum each row (which axis?) and add 50.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`((50 + steps.sum(axis=1)) > 55).mean()`

</details>

*Auto-check id:* `02-p6`  ·  *Set the variable `answer`, then run the cell.*

In [28]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('02-p6', answer)

⏳ 02-p6: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 2.7: Revenue by market

Using `prices_tbl` (3 markets × 3 crops, defined in section 4; recreate it: rows `[60,150,90]`, `[65,160,95]`, `[58,145,85]`) and the quantities `kg_sold = [1000, 400, 700]`, compute the revenue at each market with a **single matrix product**. Then return the revenue of the **best** market minus the revenue of the **worst** market.

*Expected type:* a number.

<details>
<summary>💡 Hint 1</summary>

Shapes: `(3, 3) @ (3,)` gives `(3,)`: one revenue per market.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`rev.max() - rev.min()`.

</details>

*Auto-check id:* `02-p7`  ·  *Set the variable `answer`, then run the cell.*

In [29]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('02-p7', answer)

⏳ 02-p7: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### ✍️ Problem 2.8: Explain the broadcast

In your own words (3–4 sentences): why does `price_kes + np.array([1.0, 2.5, 0.5])` fail, and what are **two different ways** to fix it depending on whether the three numbers are per-market or per-week adjustments?

✍️ *Write your answer here (double-click to edit):*



## ✅ Check your understanding

Answer in your head (or out loud!) first, *then* open the answer.

**Q1. What is the result shape of `np.ones((5, 1)) + np.ones((1, 4))`?**

<details>
<summary>Show answer</summary>

`(5, 4)`. Each size-1 dimension is stretched to match the other array.

</details>

**Q2. When would `A @ B` fail even though `A * B` works, or vice-versa?**

<details>
<summary>Show answer</summary>

`@` requires the inner dimensions to match: `(m, n) @ (n, p)`. `*` requires broadcast-compatible shapes. For example `(3, 3) * (3,)` works (broadcast) and `(3, 3) @ (3,)` also works, but `(3, 4) @ (3, 4)` fails while `(3, 4) * (3, 4)` works.

</details>

**Q3. Why is `np.random.default_rng(42)` better practice than not seeding at all?**

<details>
<summary>Show answer</summary>

Reproducibility. With a fixed seed you (and your colleagues) can regenerate the same 'random' numbers and therefore the same results.

</details>

**Q4. A column contains a single `NaN`. What does `a.mean()` return, and how do you fix it?**

<details>
<summary>Show answer</summary>

`nan`. Use `np.nanmean(a)`, or better, decide consciously how to treat the missing value (Lesson 4).

</details>

**Q5. Why does `np.diff(x)` have one element fewer than `x`?**

<details>
<summary>Show answer</summary>

It computes differences between *neighbouring* pairs: n values have n−1 gaps.

</details>

## 🧾 Cheat-sheet

| I want to… | Code |
|---|---|
| Apply math to a whole array | `np.sqrt(a)`, `np.log(a)`, `a * 2 + 1`, `np.clip(a, lo, hi)` |
| Aggregate | `a.sum(axis=0)`, `a.mean(axis=1)`, `a.std(axis=0, ddof=1)` |
| Keep the axis | `a.mean(axis=0, keepdims=True)` |
| Broadcast a column | `v[:, np.newaxis]` or `v.reshape(-1, 1)` |
| Z-score columns | `(a - a.mean(0)) / a.std(0)` |
| Matrix product | `A @ B` |
| Solve equations | `np.linalg.solve(A, b)` |
| Least squares | `np.linalg.lstsq(X, y, rcond=None)` |
| Differences / % change | `np.diff(a)`, `np.diff(a) / a[:-1] * 100` |
| NaN-safe stats | `np.nanmean`, `np.nansum`, `np.isnan` |
| IQR outliers | `q1, q3 = np.percentile(a, [25, 75])` |
| Random | `rng = np.random.default_rng(seed)`; `rng.normal`, `rng.integers`, `rng.choice` |

## ➡️ Next up
**Lesson 3; Pandas.** NumPy gave us fast arrays; pandas gives them *labels* (column names, dates, categories) and a toolbox for real, messy tables.